In [2]:
import pandas as pd
import numpy as np

In [32]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [11]:
df = pd.read_csv('../3/train.csv')

In [12]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [14]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [15]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S
...,...,...,...,...,...,...,...
106,3,female,21.0,0,0,7.6500,S
270,1,male,NaN,0,0,31.0000,S
860,3,male,41.0,2,0,14.1083,S
435,1,female,14.0,1,2,120.0000,S


In [26]:
# -------------------------------
# NUMERICAL FEATURES
# -------------------------------
# These are columns having numeric values
numerical_features = ['Age', 'Fare']

# Pipeline for numerical data preprocessing
numerical_transformer = Pipeline(steps=[
    # Step 1: Handle missing values in numerical columns
    # strategy='median' replaces NaN with median of the column
    ('imputer', SimpleImputer(strategy='median')),

    # Step 2: Scale the data
    # StandardScaler makes data mean=0 and std=1
    ('scaler', StandardScaler())
])


# -------------------------------
# CATEGORICAL FEATURES
# -------------------------------
# These are columns having text/categorical values
categorical_features = ['Embarked', 'Sex']

# Pipeline for categorical data preprocessing
categorical_transformer = Pipeline(steps=[
    # Step 1: Handle missing values in categorical columns
    # strategy='most_frequent' replaces NaN with most common value
    ('imputer', SimpleImputer(strategy='most_frequent')),

    # Step 2: Convert categories into numbers
    # OneHotEncoder creates binary columns for each category
    # handle_unknown='ignore' avoids error for new/unseen categories
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [42]:
# Creating a Machine Learning pipeline
# Step 1: preprocess the data (imputation, encoding, scaling etc.)
# Step 2: apply Logistic Regression model on processed data

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

In [43]:
# Import set_config from sklearn
# This is used to control global display settings of sklearn objects

from sklearn import set_config

# Set display mode to 'diagram'
# This makes Pipeline and models show as a visual flowchart (in Jupyter Notebook)
set_config(display='diagram')

# Show the pipeline object
# Instead of text output, it will now display a clean diagram
pipe

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [54]:
# -------------------------------
# HYPERPARAMETER TUNING (GridSearchCV)
# -------------------------------
# Goal: Automatically find the BEST settings for preprocessing + model
# instead of manually guessing what works best
    
from matplotlib.pyplot import clf


param_grid = {                 # param_grid -> parameters ka grid (list of options)

    # -------------------------------
    # NUMERICAL DATA (Age, Fare)
    # -------------------------------
    # We are testing how to fill missing values in numerical columns
    # Option 1: mean (average value)
    # Option 2: median (middle value)
    'preprocessor__num__imputer__strategy': ['mean', 'median'],

    # -------------------------------
    # CATEGORICAL DATA (Sex, Embarked)
    # -------------------------------
    # We are testing how to fill missing categorical values
    # Option 1: most_frequent (most common value)
    # Option 2: constant (fixed value like "Unknown")
    'preprocessor__cat__imputer__strategy': ['most_frequent', 'constant'],

    # -------------------------------
    # LOGISTIC REGRESSION PARAMETER
    # -------------------------------
    # C controls model strength:
    # small C → simple model (less overfitting)
    # large C → complex model (more flexible)
    'classifier__C': [0.1, 1.0, 10, 100]
}

# -------------------------------
# GRID SEARCH CV
# -------------------------------
# This will:
# 1. Try ALL combinations of parameters above
# 2. Train model for each combination
# 3. Use cross-validation (cv=10) to test performance
# 4. Select BEST performing combination automatically

grid_search = GridSearchCV(
    pipe,        # pipeline (preprocessing + model)
    param_grid, # all parameter options
    cv=10       # 10-fold cross validation for reliable results
)

In [55]:
grid_search.fit(X_train, y_train)

print(f"Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'classifier__C': 0.1, 'preprocessor__cat__imputer__strategy': 'most_frequent', 'preprocessor__num__imputer__strategy': 'mean'}


In [56]:
print(f'Internal CV Score: {grid_search.best_score_:.3f}')

Internal CV Score: 0.784


In [57]:
# -----------------------------------------
# GRIDSEARCH RESULTS ANALYSIS
# -----------------------------------------

# Convert GridSearchCV results (dictionary) into a DataFrame
# This helps us view all tested models in a tabular format
cv_results = pd.DataFrame(grid_search.cv_results_)

# Sort all results based on model performance
# "mean_test_score" = average accuracy from cross-validation
# ascending=False means highest accuracy comes on top
cv_results = cv_results.sort_values("mean_test_score", ascending=False)

# Select only important columns for understanding results:
# - classifier__C → Logistic Regression parameter (regularization strength)
# - preprocessor__cat__imputer__strategy → categorical missing value handling method
# - preprocessor__num__imputer__strategy → numerical missing value handling method
# - mean_test_score → final average accuracy score
cv_results[[
    'param_classifier__C',
    'param_preprocessor__cat__imputer__strategy',
    'param_preprocessor__num__imputer__strategy',
    'mean_test_score'
]]

,param_classifier__C,param_preprocessor__cat__imputer__strategy,param_preprocessor__num__imputer__strategy,mean_test_score
0,0.1,most_frequent,mean,0.783725
1,0.1,most_frequent,median,0.783725
2,0.1,constant,mean,0.783725
3,0.1,constant,median,0.783725
4,1.0,most_frequent,mean,0.782316
5,1.0,most_frequent,median,0.782316
6,1.0,constant,mean,0.782316
7,1.0,constant,median,0.782316
8,10.0,most_frequent,mean,0.782316
9,10.0,most_frequent,median,0.782316
